# 📘 코드 해설판 (Python 초보자용)

이 노트북은 원본 `2. Task_1.ipynb` 에 **번호(`# 1`, `# 2` …)가 붙은 코드 셀마다 한국어 해설**을 덧붙인 파일입니다.
코드 셀 바로 **위**의 `💡 [# N] 코드 설명` 칸이 해설입니다. 원본 코드와 원본 설명은 그대로입니다.

**이 노트북의 목표:** 웹 검색 결과를 참고해 질문에 답하는 RAG를 부품부터 직접 조립합니다.

```
질문 + 웹 검색 결과(HTML 5개)
        │
        ▼
 ① Retriever (검색기) : HTML → 글 → 조각 → 질문과 가장 비슷한 조각 5개 선택
        │
        ▼
 ② Reader (답변기)    : 조각 + 질문 → 프롬프트 → LLM → 답
        │
        ▼
 ③ RAG = Retriever + Reader
```

- # 1 ~ # 9 : Retriever 만들기 (직접 구현판, LlamaIndex판)
- # 10 ~ # 13 : Reader 만들기
- # 14 ~ # 15 : RAG로 합치기
- # 16 ~ # 18 : 틀린 사례 분석

#📓 TASK #1: WEB-BASED RETRIEVAL SUMMARIZATION

In this section, we will tackle the first task of the KDD Cup: Web-based Retrieval Summarization. Since the KDD Cup CRAG benchmark fundamentally focuses on RAG (Retrieval-Augmented Generation), our approach will also be based on the RAG framework.

Before building the RAG system, let’s first clarify what problem we need to solve. At first, participants receive 5 web pages per question, potentially containing relevant information. And the objective is to measure the systems' capability to identify and condense this information into accurate answers.

<br/>

<img src="https://i.imgur.com/jlNdBmD.png">

By looking at the diagram above, you will get an idea of what problem we need to solve. Additionally, since you have already reviewed the input data in previous sessions, you are well aware of the types of data you will be working with.

As you may recall, the CRAG dataset contains many challenging questions, and as we observed earlier, it is difficult for the LLM alone to solve these problems effectively. Therefore, we will explore how these types of problems can be addressed using the RAG framework.

Specifically, This practice class will be comprised of four sections.  
  
### I. Implementing a Retriever
### II. Implementing a Reader
### III. Implementing a RAG
### IV. Error case analysis

## I. Implementing a Retriever

Before building the RAG system, the first essential component we need is the **Retriever**. As you are already familiar, the retriever is a crucial element for building an effective RAG. If the retriever successfully retrieves a sufficient amount of relevant information and passes it to the LLM, the probability of the LLM generating the correct answer will significantly increase.

In the previous session, we only experimented with the default retriever provided by `LlamaIndex` and made minor adjustments, such as modifying the chunk size.   
This time, however, we will define the retriever in a more low-level manner and explore its use step by step.

This section is divided into the following four stages.

1. Preparing Python Packages
2. Implementing a Chunk Extractor
3. Implementing a Retriever
4. Implementing a Retriever with LlamaIndex





### 1. Preparing Python Packages

As always, we will start by installing and importing the necessary libraries for use.  

At this point, we will also set the values for the global variables that will be needed later. The significance of these values will be explained in detail in the following steps.


### 💡 [# 1] 코드 설명 — 라이브러리 설치

**한 줄 요약:** 실습에 필요한 라이브러리를 설치합니다.

- `os.environ.pop("PYTHONPATH", None)` : 라이브러리 충돌 방지
- `!pip install ...` : 터미널 설치 명령. 줄 끝 `\` 는 명령이 다음 줄로 이어진다는 뜻
- 실제로 쓰는 주요 라이브러리: `openai`(LLM·임베딩), `llama-index`(RAG 도구), `beautifulsoup4`(HTML 처리), `blingfire`(문장 자르기), `numpy`(숫자 계산, 다른 라이브러리와 함께 설치됨)

In [ ]:
# 1
### YOUR CODE HERE ###
import os
os.environ.pop("PYTHONPATH", None)

!pip install openai==1.55.3 llama-index llama-index-llms-openai \
    llama-index-embeddings-huggingface llama-index-readers-wikipedia wikipedia \
    packaging>=24.0 watchdog kubernetes==26.1.0 \
    sentence-transformers scikit-learn ray textwrap3 \
    streamlit==1.35.0 beautifulsoup4 blingfire \
    langchain nltk>=3.8.1 --quiet

### 💡 [# 2] 코드 설명 — 도구 불러오기와 API 키 등록

**한 줄 요약:** 앞으로 쓸 도구들을 불러오고 OpenAI API 키를 등록합니다.

- `import numpy as np` : 숫자 배열 계산 도구를 `np` 라는 별명으로
- `import bz2`, `import json` : 압축 파일 열기, JSON 다루기
- `from blingfire import text_to_sentences_and_offsets` : 문장 위치 찾기 함수
- `from typing import Any, Dict, List` : 타입 힌트용 이름들
- `from bs4 import BeautifulSoup` : HTML 처리
- `os.environ["OPENAI_API_KEY"] = "sk-..."` : ⚠️ **여기에 본인 키를 넣어야** 이후 셀이 동작합니다.
- 나머지 `llama_index` 관련 import : 인덱스, 문서, 조각내기 도구

> 🔰 `ray`, `defaultdict`, `WikipediaReader` 등은 불러오기만 하고 이 노트북에서는 쓰지 않습니다.

In [ ]:
# 2
### YOUR CODE HERE ###

import numpy as np
import ray
import bz2
import json
from blingfire import text_to_sentences_and_offsets
from collections import defaultdict
from typing import Any, Dict, List
from bs4 import BeautifulSoup
import os
import openai

os.environ["OPENAI_API_KEY"] = "sk-..." #copy your api key

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Document, get_response_synthesizer
from llama_index.readers.wikipedia import WikipediaReader
from llama_index.core.node_parser import SentenceSplitter

import textwrap

### 💡 [# 3] 코드 설명 — 설정값(상수) 정하기

**한 줄 요약:** 여러 곳에서 쓸 숫자 설정을 이름 붙여 한곳에 모아 둡니다.

- `NUM_CONTEXT_SENTENCES = 20` : 답 생성에 고려할 문장 수
- `MAX_CONTEXT_SENTENCE_LENGTH = 1000` : 조각 하나의 최대 글자 수
- `MAX_CONTEXT_REFERENCES_LENGTH = 4000` : LLM에 넘길 참고자료 전체의 최대 글자 수
- `SENTENTENCE_TRANSFORMER_BATCH_SIZE = 128` : 임베딩을 한 번에 몇 개씩 처리할지

> 🔰 변수 이름을 **대문자**로 쓰는 것은 "바꾸지 않는 설정값(상수)"이라는 파이썬 관례입니다. 숫자를 코드 곳곳에 직접 적는 것보다 이렇게 이름을 붙여 두면 나중에 한 곳만 고치면 됩니다.

In [ ]:
# 3
### YOUR CODE HERE ###

# Define the number of context sentences to consider for generating an answer.
NUM_CONTEXT_SENTENCES = 20
# Set the maximum length for each context sentence (in characters).
MAX_CONTEXT_SENTENCE_LENGTH = 1000
# Set the maximum context references length (in characters).
MAX_CONTEXT_REFERENCES_LENGTH = 4000
# Sentence Transformer Parameters
SENTENTENCE_TRANSFORMER_BATCH_SIZE = 128 # TUNE THIS VARIABLE depending on the size of your embedding model and GPU mem available

### 2. Implementing a Chunk Extractor

This time, we will define and use a `Chunk Extractor`. As you observed during the first practice session, the Chunk Extractor is a function needed to split the search results into appropriately sized pieces for use.

Since search results are essentially `HTML` files, we will first define the `parse_htmls` function to remove HTML tags. Then, we will define the `extract_chunks` function, which splits the text extracted from the HTML into chunks. To avoid losing information at the chunk boundaries, the text will be split at the sentence level.

### 💡 [# 4] 코드 설명 — HTML → 글 → 조각으로 만드는 함수 두 개

**한 줄 요약:** 검색 결과의 HTML에서 글만 뽑는 `parse_htmls` 와, 글을 문장 조각으로 자르는 `extract_chunks` 를 정의합니다.

**`parse_htmls(search_results)`**
- `for html_text in search_results:` : 검색 결과를 하나씩
- `BeautifulSoup(html_text["page_result"], features="lxml")` : HTML 해석
- `soup.get_text(" ", strip=True)` : 태그 제거 후 글만 추출
- `all_documents.append(text)` → 마지막에 `return all_documents` : 글 목록 반환

**`extract_chunks(all_documents)`**
- `if not document:` : 글이 비어 있으면 빈 문자열을 자리 채움으로 추가
- `_, offsets = text_to_sentences_and_offsets(document)` : 각 문장의 (시작, 끝) 위치 구하기. `_` 는 안 쓰는 값을 받는 이름
- `document[start:end]` : 문장 하나 잘라 내기
- `[:MAX_CONTEXT_SENTENCE_LENGTH]` : 1000자까지만
- `return all_chunks` : 조각 목록 반환

> 🔰 `#` 뒤의 영어 문장은 **주석**(사람이 읽는 메모)이며 실행되지 않습니다.

In [ ]:
# 4
### YOUR CODE HERE ###

def parse_htmls(search_results):
    all_documents = []

    # Process each HTML text from the search results to extract text content.
    for html_text in search_results:

        # Parse the HTML content using BeautifulSoup
        soup = BeautifulSoup(html_text["page_result"], features="lxml")
        text = soup.get_text(" ", strip=True)  # Use space as a separator, strip whitespaces
        all_documents.append(text)

    return all_documents

def extract_chunks(all_documents):
    # Initialize a list to hold all extracted sentences from the search results.
    all_chunks = []

    for document in all_documents:

        if not document:
            # If no document is extracted, add an empty string as a placeholder.
            all_chunks.append("")
        else:

            # Extract offsets of sentences from the document
            _, offsets = text_to_sentences_and_offsets(document)

            # Initialize a list to store sentences
            chunks = []

            # Iterate through the list of offsets and extract sentences
            for start, end in offsets:
                # Extract the sentence and limit its length
                chunk = document[start:end][:MAX_CONTEXT_SENTENCE_LENGTH]
                all_chunks.append(chunk)

    return all_chunks

Now, let’s use the two functions to process the data by loading the search results and splitting them into chunks.

Run the code below to test the example:

### 💡 [# 5] 코드 설명 — 두 함수 시험해 보기

**한 줄 요약:** 데이터 첫 건으로 "문서 전체"와 "첫 조각"의 길이·내용을 비교합니다.

- `dataset_path = "/path/to/..."` : ⚠️ **실제 데이터 파일 경로로 바꿔야 합니다.** (예: `r"C:\...\crag_task_1_dev_v4_release.jsonl.bz2"`)
- `bz2.open(dataset_path, "rt")` : 압축 파일을 글자 모드로 열기
- `item = json.loads(line)` : 한 줄(JSON) → 사전
- `parse_htmls(item["search_results"])` → `extract_chunks(...)` : 방금 만든 함수 적용
- `len(all_documents[0])` : 첫 문서의 글자 수 / `len(all_chunks[0])` : 첫 조각의 글자 수
- `print()` : 빈 줄 출력
- `break` : 첫 건만 보고 종료

In [ ]:
# 5
### YOUR CODE HERE ###

import bz2
import json

dataset_path = "/path/to/CRAG dataset/crag_task_1_dev_v4_release.jsonl.bz2"

with bz2.open(dataset_path, "rt") as file:
    for line in file:
        item = json.loads(line)

        # Get documents
        all_documents = parse_htmls(item["search_results"])

        # Get chunks
        all_chunks = extract_chunks(all_documents)

        print("=========== Document ===========")
        print("# of Document Characters: ", len(all_documents[0]))
        print()
        print(all_documents[0])
        print()
        print("=========== Chunk ===========")
        print("# of Chunk Characters: ", len(all_chunks[0]))
        print()
        print(all_chunks[0])
        print()
        break

As a result of the test, we observed that the length of the text was reduced by nearly **100 times** after splitting it into chunks compared to using the full search results. This suggests that chunks can effectively extract only the relevant parts from the `search_results` and pass them to the LLM efficiently.

Of course, we cannot guarantee that the retriever will always retrieve relevant results. However, if we use entire documents as the retrieval unit, it will take a long time to compute embeddings, and information loss may occur during that process.

Additionally, the length of the chunk characters may not exactly match the value of `MAX_CONTEXT_SENTENCE_LENGTH`. This is because we used the text_to_sentences_and_offsets function to ensure that chunks are formed without splitting sentences.

### 3. Implementing a Retriever

This time, we will implement a **Retriever** using the Chunk Extractor we defined earlier, without relying on AI frameworks like LlamaIndex.  

For this implementation, the following components are required:

<br/>

1.	**Chunk extractor**: Used to split the input search_results into chunks.
2.	**Embedding model**: Used to generate embeddings for the chunks and the query.
3.	**Similarity metric**: Measures the similarity between embeddings. We will use cosine similarity here.

Using these components, let’s implement the `BaseRetriever` with the following code:

### 💡 [# 6] 코드 설명 — 검색기(Retriever)를 직접 구현

**한 줄 요약:** 조각과 질문을 임베딩으로 바꾸고, **코사인 유사도**로 질문과 가장 비슷한 조각 k개를 골라 주는 클래스입니다.

- `def __init__(self,):` : 객체가 만들어질 때 **자동으로 한 번 실행되는** 초기화 함수. 여기서 OpenAI 클라이언트를 준비해 `self.client` 에 보관합니다.
- **`embed_text(self, texts)`** : 글 → 숫자 벡터
  - `if isinstance(texts, str): texts = [texts]` : 글이 하나(문자열)만 들어오면 리스트로 감싸 형식을 통일
  - `self.client.embeddings.create(model="text-embedding-3-small", input=texts)` : OpenAI 임베딩 모델 호출
  - `[np.array(item.embedding) for item in response.data]` : 응답에서 벡터만 꺼내 numpy 배열로
- **`retrieve(self, query, search_results, topk)`** : 검색 본체
  1. `parse_htmls` → `extract_chunks` : HTML을 조각으로
  2. `all_embeddings = self.embed_text(all_chunks)` : 모든 조각 임베딩
  3. `query_embedding = self.embed_text(query)[0]` : 질문 임베딩(결과 리스트의 첫 번째)
  4. **코사인 유사도** 계산: `np.dot(A, q) / (|A| × |q|)`
     - `np.dot` : 내적(같은 자리 숫자끼리 곱해 더함)
     - `np.linalg.norm` : 벡터의 길이. `axis=1` 은 "행(조각)마다 따로 계산"
     - 결과는 -1~1 사이 값이며 **1에 가까울수록 뜻이 비슷**합니다.
  5. `(-cosine_scores).argsort()[:topk]` : `argsort` 는 작은 값 순으로 **위치 번호**를 줍니다. 점수에 `-` 를 붙여 뒤집으면 "큰 점수 순"이 되고, 앞에서 k개를 고릅니다.
  6. `np.array(all_chunks)[top_k_indices]` : 그 위치의 조각들만 꺼냄

> 🔰 비유: 모든 조각과 질문을 지도 위의 화살표로 놓고, 질문 화살표와 **방향이 가장 비슷한** 화살표 5개를 고르는 것입니다.

In [ ]:
# 6
### YOUR CODE HERE ###

class BaseRetriever:
    def __init__(self,):
        self.client = openai.OpenAI(api_key = os.environ["OPENAI_API_KEY"])

    def embed_text(self, texts):
        """Generate embeddings using OpenAI's embedding model."""
        if isinstance(texts, str):
            texts = [texts]

        response = self.client.embeddings.create(
            model="text-embedding-3-small",
            input=texts
        )

        # Extract embeddings correctly from the response object
        embeddings = [np.array(item.embedding) for item in response.data]  # Adjust based on actual attributes
        return np.array(embeddings)

    def retrieve(self, query, search_results, topk):
        # Get documents
        all_documents = parse_htmls(search_results)

        # Get chunks
        all_chunks = extract_chunks(all_documents)

        # Generate embeddings for all chunks and the query.
        all_embeddings = self.embed_text(all_chunks)
        query_embedding = self.embed_text(query)[0]  # Single query embedding

        # Calculate cosine similarity between query and sentence embeddings, and select the top sentences.
        cosine_scores = np.dot(all_embeddings, query_embedding) / (
            np.linalg.norm(all_embeddings, axis=1) * np.linalg.norm(query_embedding)
        )
        top_k_indices = (-cosine_scores).argsort()[:topk]
        top_k_chunks = np.array(all_chunks)[top_k_indices]

        return top_k_chunks

The retriever determines the final chunks to return through three main steps.  

1.	**First Step**: The retriever takes the query, the search_results, and a variable topk (which determines how many chunks to return) as inputs. It then extracts chunks from the `search_results`.
2.	**Second Step**: The extracted chunks are converted into embeddings using an embedding model. Since the chunks are in a list format, the embedding results will also be returned as a list. At the same time, the query is also converted into an embedding.
3. **Third Step**: **Cosine similarity** between the query’s embedding and the chunks’ embeddings is calculated to determine which chunks have the highest similarity to the query.

Through this process, our `BaseRetriever` retrieves and returns the `topk` chunks with the highest similarity.  

Here is the code to verify the process:

### 💡 [# 7] 코드 설명 — 직접 만든 검색기 시험

**한 줄 요약:** 데이터 첫 건의 질문으로 상위 5개 조각을 검색해 순위와 함께 출력합니다.

- `retriever = BaseRetriever()` : 객체 생성(이때 `__init__` 실행)
- `topk = 5` : 5개 가져오기
- `retriever.retrieve(item['query'], item['search_results'], topk)` : 검색 실행
- `for rank, retrieved_result in enumerate(retrieved_results):` : `enumerate` 는 **(순번, 값)** 을 함께 줍니다. 순번은 0부터이므로 `rank+1` 로 1위부터 표시

In [ ]:
# 7
### YOUR CODE HERE ###

retriever = BaseRetriever()
topk = 5

with bz2.open(dataset_path, "rt") as file:
    for line in file:
        item = json.loads(line)
        print(f"query: {item['query']}")
        print()
        retrieved_results = retriever.retrieve(item['query'], item['search_results'], topk)
        break

print("retrieved results:")
print()
for rank, retrieved_result in enumerate(retrieved_results):
    print(f"rank {rank+1}: {retrieved_result}")
    print()

### 3. Implementing a Retriever with Llama Index

You may recall that in Day 1 practice, we defined a retriever using `LlamaIndex`.

In this exercise, we will again define a retriever using LlamaIndex. To create a retriever with `LlamaIndex`, we must first build an index. To build the index, we need to decide which data to use – in this case, we will use the `search_results`.

Follow the code below to declare the retriever:

### 💡 [# 8] 코드 설명 — LlamaIndex로 같은 검색기 만들기

**한 줄 요약:** # 6에서 손으로 만든 과정을 LlamaIndex 도구로 훨씬 짧게 구현합니다.

- `Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")` : LlamaIndex 전체에서 쓸 임베딩 모델을 지정(전역 설정)
- `__init__` : `SentenceSplitter(chunk_size=512, chunk_overlap=0)` — 512토큰씩, 겹침 없이 자르는 도구 준비
- `retrieve(self, query, search_results, topk)` :
  - `parse_htmls(search_results)` 로 얻은 글을 `Document(text=...)` 로 감싸 리스트에 모음 (빈 글이면 빈 문서)
  - `VectorStoreIndex.from_documents(documents=documents, transformations=[self.parser])` : 조각내기 + 임베딩 + 인덱스 생성
  - `base_index.as_retriever(similarity_top_k=topk)` : 상위 k개 검색기
  - `base_retriever.retrieve(query)` : 검색
  - `[retrieved_node.node.get_content().strip() for retrieved_node in retrieved_nodes]` : 검색된 각 노드에서 글만 꺼내 리스트로

> 🔰 # 6은 **문장 단위(짧은 조각)**, 여기는 **512토큰 단위(긴 조각)** 라는 차이가 있습니다. 또 질문 하나마다 인덱스를 새로 만드는 구조입니다(질문마다 검색 결과가 다르기 때문).

In [ ]:
# 8
### YOUR CODE HERE ###

from llama_index.core.schema import Document
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import VectorStoreIndex, Settings
from llama_index.embeddings.openai import OpenAIEmbedding

Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

class LlamaIndexRetriever:
  def __init__(self):
      self.parser = SentenceSplitter(chunk_size=512, chunk_overlap=0)

  def retrieve(self, query, search_results, topk):
      documents = []

      for document in parse_htmls(search_results):
        if not document:
            # If no text is extracted, add an empty string as a placeholder.
            documents.append(Document(text=""))
        else:
            documents.append(Document(text=document))

      # Split documents into chunks & Create vector index
      base_index = VectorStoreIndex.from_documents(documents = documents, transformations=[self.parser])

      # Execute query
      base_retriever = base_index.as_retriever(similarity_top_k=topk)

      retrieved_nodes = base_retriever.retrieve(query)

      retrieved_results = [retrieved_node.node.get_content().strip() for retrieved_node in retrieved_nodes]

      return retrieved_results

By leveraging an external AI framework like LlamaIndex, we can see that the code has become significantly more concise and streamlined.

Now, let’s practice using the same approach with an example to verify how it works in action!

### 💡 [# 9] 코드 설명 — LlamaIndex 검색기 시험

**한 줄 요약:** # 7과 같은 방식으로 LlamaIndex 검색기의 결과를 출력합니다.

- `retriever = LlamaIndexRetriever()` 로 바뀐 것 외에는 # 7과 같습니다.
- 여기서는 `rank` 를 그대로 출력하므로 순위가 **0부터** 표시됩니다.

> 🔰 # 7의 결과와 비교해 보세요. 조각이 더 길고 문맥이 풍부합니다.

In [ ]:
# 9
### YOUR CODE HERE ###

retriever = LlamaIndexRetriever()
topk = 5

with bz2.open(dataset_path, "rt") as file:
    for line in file:
        item = json.loads(line)
        print(f"query: {item['query']}")
        print()
        retrieved_results = retriever.retrieve(item['query'], item['search_results'], topk)
        break

print("retrieved results:")
print()
for rank, retrieved_result in enumerate(retrieved_results):
    print(f"rank {rank}: {retrieved_result}")
    print()

## II. Implementing a Reader

In this section, we will design the **Reader**.

What are the most important considerations when creating a Reader? The most crucial factor is likely the choice of LLM. Factors such as model size, performance on reasoning benchmarks, cost, and other considerations are typically part of the configuration.

However, since we have limited options for the LLMs we can use in this practice session, this will not be a consideration for us here.

So, what’s the next most important factor? **Prompt design**. It is well known that well-designed prompts lead to better results from the LLM.

Moreover, setting an appropriate prompt becomes even more critical for the CRAG dataset. In this task, the LLM must be able to answer “I don’t know” if it encounters something it is unsure about or cannot answer confidently. To achieve this, the prompt must be specifically designed to guide the LLM to behave in this manner.

Therefore, this exercise will be conducted in the following three main stages:

1. Design a Prompt Template
2. Implement a Prompt Generator
3. Implement a Reader

### 1. Design a Prompt Template

To design an effective prompt template, we need to carefully consider certain factors.

1.	The response must be generated based on the given question and references.
2.	In the CRAG benchmark, answers should not be too long or verbose. During evaluation, only the first 75 tokens are used for scoring, so the response needs to be concise.
3.	The LLM must be able to recognize questions it cannot answer and respond with “I don’t know”.

Taking these factors into account, we can draft the following `system_prompt`:

### 💡 [# 10] 코드 설명 — 시스템 프롬프트 작성

**한 줄 요약:** LLM에게 "어떻게 답해야 하는지" 알려 주는 기본 지시문을 만듭니다.

- `system_prompt = """..."""` : 여러 줄 문자열
- 지시 내용: ① 질문과 참고자료가 주어진다 ② **최대한 짧게** 답하라 ③ 참고자료에 정보가 없으면 **"I don't know"** 라고 답하라 ④ 이유 설명은 필요 없다

> 🔰 "모르면 모른다고 하라"는 지시는 LLM이 그럴듯하게 지어내는 것(**환각, hallucination**)을 줄이는 핵심 장치입니다.

In [ ]:
# 10
### YOUR CODE HERE ###

system_prompt = """
You are provided with a question and various references.
Your task is to answer the question succinctly, using the fewest words possible.
If the references do not contain the necessary information to answer the question, respond with 'I don't know'.
There is no need to explain the reasoning behind your answers.
"""

### 2. Implement a Prompt Generator

Above, we created a system prompt. Now, we need to build a prompt generator that takes the question and reference, combines them into one, and formats it so it can be passed to the LLM.

Below is an example of a `prompt_generator` that takes a question and reference, combines them for delivery to the LLM:

### 💡 [# 11] 코드 설명 — LLM 입력을 조립하는 함수

**한 줄 요약:** 질문과 검색된 조각들을 받아 LLM에 보낼 메시지 형식으로 만들어 주는 함수입니다.

- `references = ""` 로 시작해, 조각이 있으면 `"# References \n"` 제목을 붙이고 각 조각을 `- 조각내용` 형태로 한 줄씩 추가
- `references[:MAX_CONTEXT_REFERENCES_LENGTH]` : 참고자료를 4000자로 제한
- `user_message` : 참고자료 + 구분선 + "위 참고자료만 사용해 다음 질문에 답하라" + 질문
- `llm_input = [{"role": "system", ...}, {"role": "user", ...}]` : 시스템 지시와 사용자 메시지를 묶은 리스트
- `return llm_input`

In [ ]:
# 11
### YOUR CODE HERE ###

def prompt_generator(query, top_k_chunks, system_prompt):
    user_message = ""
    references = ""

    if len(top_k_chunks) > 0:
        references += "# References \n"
        # Format the top sentences as references in the model's prompt template.
        for chunk_id, chunk in enumerate(top_k_chunks):
            references += f"- {chunk.strip()}\n"

    references = references[:MAX_CONTEXT_REFERENCES_LENGTH]
    # Limit the length of references to fit the model's input size.

    user_message += f"{references}\n------\n\n"
    user_message += f"Using only the references listed above, answer the following question: \n"
    user_message += f"Question: {query}\n"

    llm_input = [
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": user_message},
    ]

    return llm_input

### 3. Implement a Reader

Now that we have created a function to generate the necessary prompts for the Reader, we will proceed to define the Reader itself and set up the components needed for RAG creation.

Follow the code below to implement it.

### 💡 [# 12] 코드 설명 — 답변기(Reader) 클래스

**한 줄 요약:** # 10의 프롬프트와 # 11의 함수를 클래스 안에 넣고, LLM을 호출해 답을 만드는 `Reader` 를 정의합니다.

- `oai_client = OpenAI()` : OpenAI 클라이언트 생성
- `__init__` : 시스템 프롬프트를 `self.system_prompt` 에 저장
- `generate_response(self, query, top_k_chunks)` :
  - `self.prompt_generator(query, top_k_chunks)` : LLM 입력 조립
  - `oai_client.chat.completions.create(model="gpt-3.5-turbo", temperature=0, messages=llm_input)` : LLM 호출 (`temperature=0` = 일관된 답)
  - `.choices[0].message.content` : 답변 글만 추출해 반환
- `prompt_generator(self, query, top_k_chunks)` : # 11과 같은 내용을 클래스 안으로 옮긴 것 (`system_prompt` 대신 `self.system_prompt` 사용)

> 🔰 중간에 `user_message` 라고만 적힌 줄은 아무 일도 하지 않는 줄입니다(지워도 됩니다).

In [ ]:
# 12
### YOUR CODE HERE ###

from openai import OpenAI

oai_client = OpenAI()

class Reader:
  def __init__(self):

    self.system_prompt = """
    You are provided with a question and various references.
    Your task is to answer the question succinctly, using the fewest words possible.
    If the references do not contain the necessary information to answer the question, respond with 'I don't know'.
    There is no need to explain the reasoning behind your answers.
    """

  def generate_response(self, query: str, top_k_chunks: list) -> str:
      """
      Generate answer from context.
      """
      llm_input = self.prompt_generator(query, top_k_chunks)
      completion = oai_client.chat.completions.create(
      model="gpt-3.5-turbo",
      temperature=0,
      messages=
      llm_input
      ).choices[0].message.content
      return completion

  def prompt_generator(self, query, top_k_chunks):
      user_message = ""
      references = ""

      if len(top_k_chunks) > 0:
          references += "# References \n"
          # Format the top sentences as references in the model's prompt template.
          for chunk_id, chunk in enumerate(top_k_chunks):
              references += f"- {chunk.strip()}\n"

      references = references[:MAX_CONTEXT_REFERENCES_LENGTH]
      # Limit the length of references to fit the model's input size.

      user_message += f"{references}\n------\n\n"
      user_message
      user_message += f"Using only the references listed above, answer the following question: \n"
      user_message += f"Question: {query}\n"

      llm_input = [
        {"role": "system", "content": self.system_prompt},
        {"role": "user", "content": user_message},
      ]

      return llm_input

Now, let’s check the results through an actual example.


### 💡 [# 13] 코드 설명 — 참고자료 없이 Reader 시험

**한 줄 요약:** 검색 결과를 **주지 않고**(빈 리스트) Reader에게 질문해 봅니다.

- `reader.generate_response(item['query'], [])` : 두 번째 인자 `[]` 가 빈 참고자료
- 질문, 정답(`ground truth`), Reader의 답을 출력

> 🔰 참고자료가 없으니 지시대로 "I don't know"라고 답하는 것이 정상입니다.

In [ ]:
# 13
### YOUR CODE HERE ###

reader = Reader()

with bz2.open(dataset_path, "rt") as file:
    for line in file:
        item = json.loads(line)
        print(f"query: {item['query']}")
        print(f"ground truth: {item['answer']}")
        print()
        answer = reader.generate_response(item['query'], [])
        break

print(f"answer: {answer}")

## III. Implementing a RAG

At this point, we have defined both the Reader and the Retriever, and we have verified their inputs and outputs.

Now, let’s combine these two components into a functional RAG system that we can use.

### 💡 [# 14] 코드 설명 — Retriever + Reader = RAG

**한 줄 요약:** 검색기와 답변기를 하나로 묶은 `RAG` 클래스를 정의합니다.

- `__init__` : `self.retriever = LlamaIndexRetriever()`, `self.reader = Reader()` — 두 부품을 내 것으로 보관
- `inference(self, query, search_results, topk)` :
  1. `self.retriever.retrieve(...)` : 관련 조각 검색
  2. `self.reader.generate_response(query, retrieved_results)` : 조각을 참고해 답 생성
  3. `return answer, retrieved_results` : 답과 검색 결과를 **함께** 반환(나중에 원인 분석에 쓰려고)

In [ ]:
# 14
### YOUR CODE HERE ###

class RAG:
    def __init__(self):
        self.retriever = LlamaIndexRetriever()
        self.reader = Reader()

    def inference(self, query, search_results, topk):
        # 1. retrieve relevant chunks
        retrieved_results = self.retriever.retrieve(query, search_results, topk)

        # 2. answer the question based on the retrieved chunks
        answer = self.reader.generate_response(query, retrieved_results)

        return answer, retrieved_results

Let’s now verify whether the RAG system we defined works as intended or not.

Using the code below, we will test the system on a total of 10 data points. You can check each result yourself and evaluate whether the RAG performs well enough.

### 💡 [# 15] 코드 설명 — RAG로 질문 10개 풀어 보기

**한 줄 요약:** 데이터 앞쪽 10개 질문에 대해 RAG의 답과 실제 정답을 나란히 출력합니다.

- `repeat = 0` : 몇 건 처리했는지 세는 변수
- `if repeat > 9: break` : 10건을 넘으면 종료
- `rag.inference(...)[0]` : 반환값 (답, 검색결과) 중 **첫 번째(답)** 만 사용
- `repeat += 1` : 한 건 처리할 때마다 1 증가

> 🔰 예측(`predicted answer`)과 정답(`ground truth answer`)을 눈으로 비교해 보세요. 맞는 것도, "I don't know"도, 틀린 것도 섞여 있을 것입니다.

In [ ]:
# 15
### YOUR CODE HERE ###

rag = RAG()
topk = 5

repeat = 0
with bz2.open(dataset_path, "rt") as file:
    for line in file:
        if repeat > 9:
          break

        item = json.loads(line)
        print(f"query: {item['query']}")
        print()
        answer = rag.inference(item['query'], item['search_results'], topk)[0]
        print(f"predicted answer: {answer}")
        print(f"ground truth answer: {item['answer']}")
        print()
        repeat += 1

## IV. Error case analysis

Some of you may be satisfied with the experimental results above, while others may not. However, few would believe that the RAG system produced the correct answer for all questions.

Therefore, before formally evaluating the RAG system, we will check which questions it answered incorrectly and try to understand why those results occurred. To do this, we need to classify the data into two categories:

1.	Questions the RAG answered correctly.
2.	Questions the RAG answered incorrectly.

Ultimately, before moving on to Task 2 in the next session, we will execute the RAG implemented for Task 1 and analyze which queries the system struggles to answer correctly.

To begin, let’s check how well the Reader alone performs on the following questions, without using search results.
<br/>  
Question: **In 2004, which animated film was recognized with the best animated feature film oscar?**.   
Answer: **Finding Nemo**
<br/>

### 💡 [# 16] 코드 설명 — 오답 사례 ① : 참고자료 없이 묻기

**한 줄 요약:** 5번 데이터(0부터 세어 6번째)만 골라, 참고자료 없이 Reader에게 물어봅니다.

- `if repeat != 5: repeat += 1; continue` : 5번이 아니면 세기만 하고 **건너뜀**. `continue` 는 "아래 코드는 생략하고 다음 반복으로"라는 뜻
- 5번에 도달하면 질문·답·정답 출력 후 `break`

In [ ]:
# 16
### YOUR CODE HERE ###
repeat = 0
with bz2.open(dataset_path, "rt") as file:
    for line in file:
        if repeat != 5:
          repeat += 1
          continue

        item = json.loads(line)
        print(f"query: {item['query']}")
        print()
        answer = reader.generate_response(item['query'], [])
        print(f"predicted answer: {answer}")
        print(f"ground truth answer: {item['answer']}")
        print()
        repeat += 1
        break

Although the correct answer is **“Finding Nemo”**, the model generated the incorrect answer, **“The Incredibles”**.

Next, let’s check the generated result when search results are utilized.


### 💡 [# 17] 코드 설명 — 오답 사례 ② : 같은 질문을 RAG로

**한 줄 요약:** 같은 5번 질문을 RAG로 풀고, 어떤 조각이 검색되었는지까지 출력합니다.

- `answer, retrieved_results = rag.inference(...)` : 답과 검색 결과를 각각 받음
- `enumerate(retrieved_results)` 로 검색된 조각을 순번과 함께 출력

> 🔰 검색된 조각에 정답("Finding Nemo")의 근거가 있는지 확인해 보세요. **검색이 틀리면 답도 틀립니다.** 틀린 정보를 근거로 자신 있게 틀린 답을 하는 것이 RAG의 대표적인 실패 유형입니다.

In [ ]:
# 17
rag = RAG()
topk = 5

repeat = 0
with bz2.open(dataset_path, "rt") as file:
    for line in file:
        if repeat != 5:
          repeat += 1
          continue

        item = json.loads(line)
        print(f"query: {item['query']}")
        print()
        answer, retrieved_results = rag.inference(item['query'], item['search_results'], topk)
        print(f"predicted answer: {answer}")
        print(f"ground truth answer: {item['answer']}")
        print()
        print("retrieved results:")
        for rank, retrieved_result in enumerate(retrieved_results):
            print(f"{rank}: {retrieved_result}")
        print()
        repeat += 1
        break

The following queries focus on retrieving information related to finance.

Such information is typically stored in structured data formats, such as tables or knowledge graphs. However, unstructured data sources, like web search results, often overlook the structural information inherent in tables or knowledge graphs, making it challenging to extract specific information efficiently.

For instance, financial data such as Microsoft's ex-dividend date, P/E ratio, or earnings per share is usually presented in numeric, date, or tabular formats. In contrast, text-based data lacks the structured representation found in tables, making it harder to leverage such information.

Let us explore whether RAG (Retrieval-Augmented Generation) can effectively answer the following queries using only web search results.

<br/>  
Question: **What is the ex-dividend date of microsoft in the 1st qtr of 2024**.   
Answer: **The ex-dividend date of microsoft in the 1st qtr of 2024 is feb 14, 2024**
<br/>

<br/>  
Question: **I'm looking for the p/e ratio of dks. would you happen to know what it is?**.   
Answer: **13.75**
<br/>

<br/>  
Question: **What's auph's earnings per share?**.   
Answer: **0.4**
<br/>


### 💡 [# 18] 코드 설명 — 오답 사례 ③ : 금융 질문들

**한 줄 요약:** 14, 53, 64번 데이터(금융 관련 질문)만 골라 RAG로 풀어 봅니다.

- `if repeat >= 65: break` : 65번째부터는 볼 필요 없으니 종료
- `if repeat not in [14, 53, 64]:` : 번호가 이 목록에 **없으면** 건너뜀
- 나머지 출력은 # 17과 같습니다.

> 🔰 주가·배당처럼 **시시각각 변하는 숫자**는 웹 검색 결과만으로는 정확히 답하기 어렵습니다. 그래서 다음 노트북(Task 2)에서 **지식그래프(KG) API**를 추가로 사용합니다.

In [ ]:
# 18
import bz2
import json


rag = RAG()
topk = 5

repeat = 0

with bz2.open(dataset_path, "rt") as file:
    for line in file:
        if repeat >= 65:
          break
        if repeat not in [14, 53, 64]:
          repeat += 1
          continue

        item = json.loads(line)
        print(f"query: {item['query']}")
        print()
        answer, retrieved_results = rag.inference(item['query'], item['search_results'], topk)
        print(f"predicted answer: {answer}")
        print(f"ground truth answer: {item['answer']}")
        print()
        print("retrieved results:")
        for rank, retrieved_result in enumerate(retrieved_results):
            print(f"{rank}: {retrieved_result}")
        print()
        repeat += 1